# P2 — Day-Disjoint Split

Notebook chuẩn bị dữ liệu CICIDS2017 đã xử lý cho P2: Train gồm Monday–Wednesday, Test gồm Thursday–Friday. Sau audit, giữ nguyên Test và loại khỏi Train những dòng có hash 78 feature xuất hiện ở Test. Sau khi duyệt split, chia riêng Train thành Train nội bộ và Validation theo nhóm feature. Logistic, Decision Tree, Random Forest và XGBoost đã hoàn tất Validation/Train/Test; workflow MLP vừa được chuẩn bị, chưa chạy trên CICIDS2017.

**Giới hạn:** Quy tắc lọc có sử dụng feature Test để chọn dòng Train; cần nêu rõ điều này khi diễn giải. Test P2 đã được xem trong các thí nghiệm cũ, không phải holdout chưa xem. Ngày và họ tấn công cùng thay đổi giữa Train/Test, nên không thể quy khác biệt chỉ cho ngày.

Mỗi model chọn cấu hình trên Validation, fit lại trên toàn Train P2 rồi mới đánh giá Test ở cell riêng. Chạy từng bước, xem output và dừng trước khi mở khóa bước tiếp theo.

## 1. Thiết lập và preflight

Tìm dataset và helper P2 trong project; kiểm tra header, mẫu nhỏ, RAM và đĩa. Chưa tạo split hoặc sửa dữ liệu. Chạy hai cell theo thứ tự, gửi kết quả preflight rồi dừng.

In [1]:
from pathlib import Path
from datetime import datetime
import importlib
import sys
import uuid
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_split_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy project và helper P2.')
dataset_path = project_root / 'Dataset' / 'Processed' / 'cicids2017_clean.csv.gz'
if not dataset_path.is_file():
    raise FileNotFoundError(f'Không tìm thấy dataset processed: {dataset_path}')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_split_workflow
importlib.reload(p2_split_workflow)
run_id = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
p2_run_dir = dataset_path.parent / 'P2_Day_Disjoint' / run_id
print(f'Dataset: {dataset_path}')
print(f'Run P2 dự kiến: {p2_run_dir}')


Dataset: d:\HUST\GR2\NIDS-ML\Dataset\Processed\cicids2017_clean.csv.gz
Run P2 dự kiến: d:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17


In [2]:
p2_context = p2_split_workflow.preflight(dataset_path)
print(f"Schema: {len(p2_context['features'])} feature và 4 metadata.")


RAM khả dụng: 4.41 GiB; chunk: 20,000 dòng.
Đĩa trống: 67.12 GiB; ước tính: 0.66 GiB.
Ước tính tài nguyên không bảo đảm mức sử dụng thực tế.
Schema: 78 feature và 4 metadata.


## 2. Audit ngày, nhãn và overlap

Đọc tuần tự dataset theo chunk, kiểm tra schema, nhãn, giá trị hữu hạn và ngày theo file nguồn. Thống kê theo ngày/nhãn và đếm các nhóm hash xuất hiện ở cả Train/Test, sau đó thống kê số dòng bị ảnh hưởng theo nguồn và nhãn. Kết quả audit và SQLite được lưu trong run mới; chưa loại dòng hay tạo split. Hash 64-bit có thể va chạm; nhóm hash trùng không chứng minh là cùng flow.

Chạy sau khi xem preflight; gửi các bảng audit rồi dừng trước mục 3.

In [3]:
p2_audit = p2_split_workflow.audit_dataset(dataset_path, p2_run_dir, p2_context)
display(p2_audit['day_audit'])
display(p2_audit['partition_audit'])
display(p2_audit['overlap_audit'])
print(f'Audit P2: {p2_run_dir}')


Nhóm hash có mặt ở cả hai phía: 23,701.


,CaptureDay,BinaryLabel,Rows
0,Monday,Normal,502650
1,Monday,Attack,0
2,Tuesday,Normal,412476
3,Tuesday,Attack,9150
4,Wednesday,Normal,416736
5,Wednesday,Attack,193756
6,Thursday,Normal,414790
7,Thursday,Attack,2179
8,Friday,Normal,400247
9,Friday,Attack,220656


,Partition,Normal,Attack,Rows,UniqueFeatureGroups
0,Train,1331862,202906,1534768,1526430
1,Test,815037,222835,1037872,1017372


,Partition,CaptureDay,SourceFile,BinaryLabel,Rows
0,Test,Friday,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Normal,3720
1,Test,Friday,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Normal,6549
2,Test,Friday,Friday-WorkingHours-Morning.pcap_ISCX.csv,Normal,8864
3,Test,Thursday,Thursday-WorkingHours-Afternoon-Infilteration....,Normal,9253
4,Test,Thursday,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Normal,8037
5,Train,Monday,Monday-WorkingHours.pcap_ISCX.csv,Normal,257
6,Train,Tuesday,Tuesday-WorkingHours.pcap_ISCX.csv,Normal,11620
7,Train,Wednesday,Wednesday-workingHours.pcap_ISCX.csv,Attack,66
8,Train,Wednesday,Wednesday-workingHours.pcap_ISCX.csv,Normal,16438


Audit P2: d:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17


## 3. Tạo Train/Test P2

Sau khi duyệt audit, bật cờ để giữ toàn bộ Test và loại các dòng Train có hash trùng Test. Cell ghi CSV gzip, kiểm tra counts, ngày và overlap sau khi đọc lại file; đối chiếu nội dung đã ghi với dữ liệu được chọn. Lưu manifest và bảng dòng bị loại, không sửa dataset nguồn hoặc ghi đè artifact.

Xem bảng Train/Test và các dòng bị loại rồi dừng trước mục 4. Các cell 1–3 cần chạy trong cùng kernel để giữ đường dẫn run.

In [4]:
RUN_P2_SPLIT = True 
if not RUN_P2_SPLIT:
    raise RuntimeError('Chưa duyệt audit P2 để tạo split.')
p2_split = p2_split_workflow.create_split(p2_run_dir)
display(p2_split['partition_audit'])
display(p2_split['removed_audit'])
print(f"Số nhóm hash giao sau lọc: {p2_split['manifest']['shared_feature_groups_after_filtering']}")


,Partition,Rows,Normal,Attack
0,Train,1506387,1303547,202840
1,Test,1037872,815037,222835


,CaptureDay,SourceFile,BinaryLabel,RowsRemoved
0,Monday,Monday-WorkingHours.pcap_ISCX.csv,Normal,257
1,Tuesday,Tuesday-WorkingHours.pcap_ISCX.csv,Normal,11620
2,Wednesday,Wednesday-workingHours.pcap_ISCX.csv,Attack,66
3,Wednesday,Wednesday-workingHours.pcap_ISCX.csv,Normal,16438


Số nhóm hash giao sau lọc: 0


## 4. Chia Validation nội bộ từ Train P2

Sau khi duyệt split, bật cờ để chia Train thành Train nội bộ và Validation bằng `GroupShuffleSplit`, mục tiêu 20% nhóm feature cho Validation, `random_state=42`. Các dòng cùng hash được giữ trong một phần; tỷ lệ dòng và nhãn thực tế có thể khác 20%. Cell đọc Train, không đọc Test trong bước này; tuy nhiên Train nguồn đã được lọc bằng feature Test ở mục 3.

Xem audit theo partition, ngày và nhãn gốc rồi dừng. Validation vẫn lấy từ Monday–Wednesday, không phải đánh giá trên ngày hay họ tấn công chưa thấy. Sau khi duyệt audit, các cell Logistic phía dưới mới bắt đầu huấn luyện.

In [8]:
RUN_P2_VALIDATION = True
if not RUN_P2_VALIDATION:
    raise RuntimeError('Chưa duyệt split P2 để tạo Validation.')
p2_validation = p2_split_workflow.create_validation_split(p2_run_dir)
display(p2_validation['partition_audit'])
display(p2_validation['day_label_audit'])
display(p2_validation['original_label_audit'])
print(f"Validation run: {p2_validation['run_dir']}")


,Partition,Rows,Normal,Attack,AttackRate,UniqueFeatureGroups
0,TrainInternal,1205099,1042732,162367,0.134733,1202183
1,Validation,301288,260815,40473,0.134333,300546


,Partition,CaptureDay,BinaryLabel,Rows
0,TrainInternal,Monday,Normal,401779
1,TrainInternal,Tuesday,Attack,7312
2,TrainInternal,Tuesday,Normal,320759
3,TrainInternal,Wednesday,Attack,155055
4,TrainInternal,Wednesday,Normal,320194
5,Validation,Monday,Normal,100614
6,Validation,Tuesday,Attack,1838
7,Validation,Tuesday,Normal,80097
8,Validation,Wednesday,Attack,38635
9,Validation,Wednesday,Normal,80104


,Partition,OriginalLabel,BinaryLabel,Rows
0,TrainInternal,BENIGN,Normal,1042732
1,TrainInternal,DoS GoldenEye,Attack,8205
2,TrainInternal,DoS Hulk,Attack,138431
3,TrainInternal,DoS Slowhttptest,Attack,4156
4,TrainInternal,DoS slowloris,Attack,4255
5,TrainInternal,FTP-Patator,Attack,4737
6,TrainInternal,Heartbleed,Attack,8
7,TrainInternal,SSH-Patator,Attack,2575
8,Validation,BENIGN,Normal,260815
9,Validation,DoS GoldenEye,Attack,2081


Validation run: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b


## 5. Logistic Regression — chọn C trên Validation

Quy trình giống P1: so sánh `C=0.1, 1, 10` trên Train nội bộ/Validation; mỗi cấu hình dùng `Pipeline(StandardScaler, LogisticRegression(saga))`, nên scaler chỉ được fit trên Train nội bộ. Chọn Macro-F1 cao nhất; nếu bằng nhau chính xác thì chọn C nhỏ hơn. Cell tự tìm Validation P2 hợp lệ và tạo cache theo chunk. Test không được đọc để chọn cấu hình, dù feature Test đã được dùng lọc Train ở bước chia P2.

Bật cờ và chạy riêng cell code này. Gửi bảng kết quả để xem xét; dừng trước bước fit toàn Train.

In [9]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_logistic_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy helper Logistic P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)

RUN_P2_LOGISTIC_VALIDATION = True
if not RUN_P2_LOGISTIC_VALIDATION:
    raise RuntimeError('Chưa duyệt để so sánh Logistic trên Validation P2.')
p2_logistic_validation_dir = p2_logistic_workflow.latest_validation_run_dir()
p2_logistic_validation_results = p2_logistic_workflow.compare_validation(p2_logistic_validation_dir)
display(p2_logistic_validation_results)


Cache TrainInternal: 1,205,099 dòng × 78 feature.
Đang so sánh Logistic trên Validation: C=0.1
Peak RAM ước tính: 1.40 GiB; RAM khả dụng: 7.93 GiB.
Đang so sánh Logistic trên Validation: C=1.0
Peak RAM ước tính: 1.40 GiB; RAM khả dụng: 7.39 GiB.
Đang so sánh Logistic trên Validation: C=10.0
Peak RAM ước tính: 1.40 GiB; RAM khả dụng: 7.00 GiB.
C được chọn theo Macro-F1 Validation: 1.0


,C,fit_seconds,n_iter,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,0.1,48.863619,38,301288,258651,2164,7974,32499,0.966351,0.937570,0.80298,0.865071,0.922925,0.008297,0.19702
1,1.0,48.163281,38,301288,258652,2163,7974,32499,0.966354,0.937597,0.80298,0.865083,0.922932,0.008293,0.19702
2,10.0,50.721509,39,301288,258651,2164,7974,32499,0.966351,0.937570,0.80298,0.865071,0.922925,0.008297,0.19702


## 6. Fit pipeline Logistic trên toàn Train P2

Chỉ chạy sau khi đã xem và duyệt bảng Validation. Helper lấy C đã chọn, đọc riêng `p2_train.csv.gz` (đã lọc overlap với Test), fit lại `StandardScaler + LogisticRegression` trên toàn bộ Train P2 và lưu model cùng training summary. Cell không đọc Test; không ghi đè artifact đã tồn tại.

Bật cờ và chạy riêng cell này. Kiểm tra summary và dừng trước khi đánh giá Test.

In [10]:
from pathlib import Path
import importlib
import sys

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_logistic_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy helper Logistic P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)

RUN_P2_LOGISTIC_FINAL = True
if not RUN_P2_LOGISTIC_FINAL:
    raise RuntimeError('Chưa duyệt để fit Logistic trên toàn Train P2.')
p2_logistic_validation_dir = p2_logistic_workflow.latest_validation_run_dir(
    required_artifacts=('p2_logistic_validation_selection.json',)
)
p2_logistic_model_path = p2_logistic_workflow.fit_final(p2_logistic_validation_dir)
print(f"Model đã lưu: {p2_logistic_model_path}")


Peak RAM ước tính: 1.75 GiB; RAM khả dụng: 7.17 GiB.
Đã lưu pipeline Logistic P2: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_logistic_selected_pipeline.joblib
Model đã lưu: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_logistic_selected_pipeline.joblib


## 7. Đánh giá Logistic trên Test P2

Cell này chỉ chạy sau khi đã fit và duyệt model cuối. Đọc `p2_test.csv.gz` để dự đoán và tính Accuracy, Precision/Recall/F1 Attack, Macro-F1, FPR, FNR và confusion matrix; tổng số dự đoán phải khớp audit. Test P2 đã được dùng lọc Train và đã xem trong thí nghiệm cũ, vì vậy kết quả chỉ là đánh giá theo protocol, không phải holdout chưa từng xem và không được dùng để chỉnh cấu hình.

Bật cờ và chạy một lần; cell không ghi đè metrics đã tồn tại.

In [11]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_logistic_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy helper Logistic P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)

RUN_P2_LOGISTIC_TEST = True
if not RUN_P2_LOGISTIC_TEST:
    raise RuntimeError('Chưa duyệt để đánh giá Logistic trên Test P2.')
p2_logistic_validation_dir = p2_logistic_workflow.latest_validation_run_dir(
    required_artifacts=(
        'p2_logistic_validation_selection.json',
        'p2_logistic_selected_pipeline.joblib',
        'p2_logistic_selected_training.json',
    )
)
p2_logistic_test_results = p2_logistic_workflow.evaluate_test(p2_logistic_validation_dir)
display(p2_logistic_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,1037872,806012,9025,141375,81460,0.855088,0.90026,0.365562,0.51998,0.717321,0.011073,0.634438


## 8. Chọn cấu hình Decision Tree trên Validation P2

So sánh năm cấu hình Gini với cặp `(max_depth, min_samples_leaf)` là `(12,10)`, `(16,10)`, `(20,10)`, `(20,25)`, `(20,50)`. Cố định `max_leaf_nodes=50.000`, `class_weight=None`, `random_state=42`; chọn theo Macro-F1, nếu hòa chính xác thì ưu tiên cây nông hơn rồi `min_samples_leaf` lớn hơn.

Cell chỉ fit trên TrainInternal và đánh giá Validation; không đọc các dòng Test. Mở gate sau khi đã xem các kết quả Logistic và duyệt bắt đầu Decision Tree, chạy riêng mục này rồi dừng để xem bảng.

In [13]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_decision_tree_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow Decision Tree P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_decision_tree_workflow
importlib.reload(p2_decision_tree_workflow)

RUN_P2_DECISION_TREE_VALIDATION = True
if not RUN_P2_DECISION_TREE_VALIDATION:
    raise RuntimeError('Chưa duyệt để chọn cấu hình Decision Tree P2 trên Validation.')
p2_decision_tree_validation_dir = p2_decision_tree_workflow.latest_validation_run_dir()
p2_decision_tree_validation_results = p2_decision_tree_workflow.compare_validation(p2_decision_tree_validation_dir)
display(p2_decision_tree_validation_results)


Đã dùng cache đã xác minh: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_logistic_training_cache
Đang fit Decision Tree P2 trên TrainInternal: max_depth=12, min_samples_leaf=10
Peak RAM ước tính: 1.56 GiB; RAM khả dụng: 4.38 GiB.
Đang fit Decision Tree P2 trên TrainInternal: max_depth=16, min_samples_leaf=10
Peak RAM ước tính: 1.56 GiB; RAM khả dụng: 4.12 GiB.
Đang fit Decision Tree P2 trên TrainInternal: max_depth=20, min_samples_leaf=10
Peak RAM ước tính: 1.56 GiB; RAM khả dụng: 4.28 GiB.
Đang fit Decision Tree P2 trên TrainInternal: max_depth=20, min_samples_leaf=25
Peak RAM ước tính: 1.56 GiB; RAM khả dụng: 5.19 GiB.
Đang fit Decision Tree P2 trên TrainInternal: max_depth=20, min_samples_leaf=50
Peak RAM ước tính: 1.56 GiB; RAM khả dụng: 5.09 GiB.
Decision Tree P2 được chọn theo Macro-F1: {'criterion': 'gini', 'max_leaf_nodes': 50000, 'class_weight': None, 'random_state': 42, 'max_depth': 20, 'min_samples_

,criterion,max_leaf_nodes,class_weight,random_state,max_depth,min_samples_leaf,fit_seconds,observed_depth,leaf_count,rows,...,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,gini,50000,None,42,12,10,62.303996,12,93,301288,...,383,269,40204,0.997836,0.990563,0.993354,0.991957,0.995353,0.001468,0.006646
1,gini,50000,None,42,16,10,79.651280,16,180,301288,...,126,57,40416,0.999393,0.996892,0.998592,0.997741,0.998695,0.000483,0.001408
2,gini,50000,None,42,20,10,83.600747,20,231,301288,...,60,56,40417,0.999615,0.998518,0.998616,0.998567,0.999172,0.000230,0.001384
3,gini,50000,None,42,20,25,80.465014,20,193,301288,...,66,75,40398,0.999532,0.998369,0.998147,0.998258,0.998994,0.000253,0.001853
4,gini,50000,None,42,20,50,77.931268,20,172,301288,...,108,106,40367,0.999290,0.997332,0.997381,0.997356,0.998473,0.000414,0.002619


## 9. Fit Decision Tree cuối trên toàn Train P2

Chỉ chạy sau khi đã xem và duyệt cấu hình mục 8. Workflow đọc lựa chọn đã lưu, xác minh manifest rồi fit trên toàn Train P2; không đọc Test. Model và training summary được lưu riêng, không ghi đè artifact.

In [14]:
from pathlib import Path
import importlib
import sys

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_decision_tree_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow Decision Tree P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_decision_tree_workflow
importlib.reload(p2_decision_tree_workflow)

RUN_P2_DECISION_TREE_FINAL = True
if not RUN_P2_DECISION_TREE_FINAL:
    raise RuntimeError('Chưa duyệt để fit Decision Tree trên toàn Train P2.')
p2_decision_tree_validation_dir = p2_decision_tree_workflow.latest_validation_run_dir(
    required_artifacts=('p2_decision_tree_validation_selection.json',)
)
p2_decision_tree_model_path = p2_decision_tree_workflow.fit_final(p2_decision_tree_validation_dir)
print(f'Model đã lưu: {p2_decision_tree_model_path}')


Peak RAM ước tính: 1.83 GiB; RAM khả dụng: 3.41 GiB.
Đã lưu Decision Tree P2: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_decision_tree_selected_model.joblib
Model đã lưu: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_decision_tree_selected_model.joblib


## 10. Đánh giá Decision Tree trên Test P2

Chỉ chạy sau khi model cuối mục 9 đã fit và được duyệt. Cell dự đoán Test và báo Accuracy, Precision/Recall/F1 Attack, Macro-F1, FPR, FNR cùng confusion matrix. Test P2 đã được dùng lọc Train và đã xem trong thí nghiệm cũ; kết quả chỉ mô tả protocol, không dùng để chỉnh cấu hình.

In [15]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_decision_tree_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow Decision Tree P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_decision_tree_workflow
importlib.reload(p2_decision_tree_workflow)

RUN_P2_DECISION_TREE_TEST = True
if not RUN_P2_DECISION_TREE_TEST:
    raise RuntimeError('Chưa duyệt để đánh giá Decision Tree trên Test P2.')
p2_decision_tree_validation_dir = p2_decision_tree_workflow.latest_validation_run_dir(
    required_artifacts=(
        'p2_decision_tree_validation_selection.json',
        'p2_decision_tree_selected_model.joblib',
        'p2_decision_tree_selected_training.json',
    )
)
p2_decision_tree_test_results = p2_decision_tree_workflow.evaluate_test(p2_decision_tree_validation_dir)
display(p2_decision_tree_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,1037872,814721,316,138238,84597,0.866502,0.996279,0.37964,0.549781,0.735707,0.000388,0.62036


## 11. Chọn cấu hình Random Forest trên Validation P2

So sánh năm cặp `(max_depth, min_samples_leaf)`: `(12,20)`, `(16,20)`, `(20,20)`, `(20,10)`, `(20,50)`. Như P1, mỗi rừng có 50 cây, Gini, tối đa 10.000 lá/cây, `max_features='sqrt'`, `bootstrap=True`, `max_samples=0.25`, `n_jobs=1`, `class_weight=None`, seed 42. Mỗi cây lấy mẫu bootstrap có hoàn lại bằng 25% số dòng TrainInternal. Chọn Macro-F1 trên Validation; nếu hòa chính xác, ưu tiên depth nhỏ hơn rồi leaf lớn hơn.

Fit lần lượt trên TrainInternal và đánh giá Validation; không dùng Test chọn cấu hình. Bật gate, chạy riêng mục này và gửi bảng kết quả, rồi dừng trước mục 12. Ước tính RAM không bảo đảm peak thực tế.

In [2]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_random_forest_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow Random Forest P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_random_forest_workflow
importlib.reload(p2_random_forest_workflow)

RUN_P2_RANDOM_FOREST_VALIDATION = True
if not RUN_P2_RANDOM_FOREST_VALIDATION:
    raise RuntimeError('Chưa duyệt để chọn cấu hình Random Forest P2 trên Validation.')
p2_random_forest_validation_dir = p2_random_forest_workflow.latest_validation_run_dir()
p2_random_forest_validation_results = p2_random_forest_workflow.compare_validation(p2_random_forest_validation_dir)
display(p2_random_forest_validation_results)


Peak RAM ước tính: 1.67 GiB; RAM khả dụng: 3.44 GiB.
Đã dùng cache đã xác minh: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_logistic_training_cache
Đang fit Random Forest P2 trên TrainInternal: max_depth=12, min_samples_leaf=20
Peak RAM ước tính: 1.67 GiB; RAM khả dụng: 3.45 GiB.
Đang fit Random Forest P2 trên TrainInternal: max_depth=16, min_samples_leaf=20
Peak RAM ước tính: 1.67 GiB; RAM khả dụng: 4.21 GiB.
Đang fit Random Forest P2 trên TrainInternal: max_depth=20, min_samples_leaf=20
Peak RAM ước tính: 1.67 GiB; RAM khả dụng: 4.44 GiB.
Đang fit Random Forest P2 trên TrainInternal: max_depth=20, min_samples_leaf=10
Peak RAM ước tính: 1.67 GiB; RAM khả dụng: 4.26 GiB.
Đang fit Random Forest P2 trên TrainInternal: max_depth=20, min_samples_leaf=50
Peak RAM ước tính: 1.67 GiB; RAM khả dụng: 4.35 GiB.
Random Forest P2 được chọn theo Macro-F1: {'criterion': 'gini', 'max_leaf_nodes': 10000, 'n_estimators': 50,

,criterion,max_leaf_nodes,n_estimators,max_features,bootstrap,max_samples,n_jobs,class_weight,random_state,max_depth,...,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,gini,10000,50,sqrt,True,0.25,1,None,42,12,...,12,635,39838,0.997853,0.999699,0.984311,0.991945,0.995353,0.000046,0.015689
1,gini,10000,50,sqrt,True,0.25,1,None,42,16,...,24,371,40102,0.998689,0.999402,0.990833,0.995099,0.997171,0.000092,0.009167
2,gini,10000,50,sqrt,True,0.25,1,None,42,20,...,26,288,40185,0.998958,0.999353,0.992884,0.996108,0.997753,0.000100,0.007116
3,gini,10000,50,sqrt,True,0.25,1,None,42,20,...,33,216,40257,0.999174,0.999181,0.994663,0.996917,0.998220,0.000127,0.005337
4,gini,10000,50,sqrt,True,0.25,1,None,42,20,...,55,459,40014,0.998294,0.998627,0.988659,0.993618,0.996317,0.000211,0.011341


## 12. Fit Random Forest cuối trên toàn Train P2

Chỉ chạy sau khi đã xem và duyệt cấu hình mục 11. Fit cấu hình đã chọn trên toàn Train P2, lưu model và training summary; không dùng Test để fit, không ghi đè artifact.

In [3]:
from pathlib import Path
import importlib
import sys

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_random_forest_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow Random Forest P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_random_forest_workflow
importlib.reload(p2_random_forest_workflow)

RUN_P2_RANDOM_FOREST_FINAL = True
if not RUN_P2_RANDOM_FOREST_FINAL:
    raise RuntimeError('Chưa duyệt để fit Random Forest trên toàn Train P2.')
p2_random_forest_validation_dir = p2_random_forest_workflow.latest_validation_run_dir(
    required_artifacts=('p2_random_forest_validation_selection.json',)
)
p2_random_forest_model_path = p2_random_forest_workflow.fit_final(p2_random_forest_validation_dir)
print(f'Model đã lưu: {p2_random_forest_model_path}')


Peak RAM ước tính: 1.93 GiB; RAM khả dụng: 4.09 GiB.
Peak RAM ước tính: 1.93 GiB; RAM khả dụng: 3.78 GiB.
Đã lưu Random Forest P2: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_random_forest_selected_model.joblib
Model đã lưu: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_random_forest_selected_model.joblib


## 13. Đánh giá Random Forest trên Test P2

Chỉ chạy sau khi model cuối mục 12 đã fit và được duyệt. Đánh giá Accuracy, Precision/Recall/F1 Attack, Macro-F1, FPR, FNR và confusion matrix. Test P2 đã được dùng lọc Train và đã xem trong thí nghiệm cũ; kết quả mô tả protocol, không dùng để chỉnh cấu hình.

In [4]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_random_forest_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow Random Forest P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_random_forest_workflow
importlib.reload(p2_random_forest_workflow)

RUN_P2_RANDOM_FOREST_TEST = True
if not RUN_P2_RANDOM_FOREST_TEST:
    raise RuntimeError('Chưa duyệt để đánh giá Random Forest trên Test P2.')
p2_random_forest_validation_dir = p2_random_forest_workflow.latest_validation_run_dir(
    required_artifacts=(
        'p2_random_forest_validation_selection.json',
        'p2_random_forest_selected_model.joblib',
        'p2_random_forest_selected_training.json',
    )
)
p2_random_forest_test_results = p2_random_forest_workflow.evaluate_test(p2_random_forest_validation_dir)
display(p2_random_forest_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,1037872,814900,137,141667,81168,0.86337,0.998315,0.364252,0.533754,0.726856,0.000168,0.635748


## 14. Chọn cấu hình XGBoost trên Validation P2

Thử năm cấu hình `(max_depth, n_estimators, min_child_weight)`: `(4,150,1)`, `(6,150,1)`, `(8,150,1)`, `(6,250,1)`, `(6,150,5)`. Như P1, dùng CPU, `hist`, `max_bin=128`, `learning_rate=0.1`, `subsample=0.8`, `colsample_bytree=0.8`, 4 luồng, seed 42, không scaler và không early stopping. Chọn Macro-F1 trên Validation; nếu hòa chính xác, ưu tiên depth nhỏ hơn, ít cây hơn, rồi `min_child_weight` lớn hơn.

Fit trên TrainInternal và đánh giá Validation; không đọc các dòng Test để chọn cấu hình. Helper vẫn kiểm tra header Test; feature Test đã được dùng lọc Train ở bước split. Trước mỗi lần fit cần tối thiểu 4 GiB RAM khả dụng và 3 GiB đĩa trống; đây không phải giới hạn peak RAM thực tế. Bật gate, chạy riêng mục này và gửi bảng kết quả, rồi dừng trước mục 15.

In [1]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_xgboost_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow XGBoost P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_xgboost_workflow
importlib.reload(p2_xgboost_workflow)

RUN_P2_XGBOOST_VALIDATION = True
if not RUN_P2_XGBOOST_VALIDATION:
    raise RuntimeError('Chưa duyệt để chọn cấu hình XGBoost P2 trên Validation.')
p2_xgboost_validation_dir = p2_xgboost_workflow.latest_validation_run_dir()
p2_xgboost_validation_results = p2_xgboost_workflow.compare_validation(p2_xgboost_validation_dir)
display(p2_xgboost_validation_results)


RAM khả dụng: 7.40 GiB; đĩa trống: 66.21 GiB. Đây không phải giới hạn peak RAM thực tế.
Đã dùng cache đã xác minh: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_logistic_training_cache
Đang fit XGBoost P2 trên TrainInternal: {'max_depth': 4, 'n_estimators': 150, 'min_child_weight': 1}
RAM khả dụng: 7.39 GiB; đĩa trống: 66.21 GiB. Đây không phải giới hạn peak RAM thực tế.
Đang fit XGBoost P2 trên TrainInternal: {'max_depth': 6, 'n_estimators': 150, 'min_child_weight': 1}
RAM khả dụng: 7.20 GiB; đĩa trống: 66.21 GiB. Đây không phải giới hạn peak RAM thực tế.
Đang fit XGBoost P2 trên TrainInternal: {'max_depth': 8, 'n_estimators': 150, 'min_child_weight': 1}
RAM khả dụng: 7.10 GiB; đĩa trống: 66.21 GiB. Đây không phải giới hạn peak RAM thực tế.
Đang fit XGBoost P2 trên TrainInternal: {'max_depth': 6, 'n_estimators': 250, 'min_child_weight': 1}
RAM khả dụng: 7.15 GiB; đĩa trống: 66.21 GiB. Đây không phải giới hạn 

,max_depth,n_estimators,min_child_weight,fit_seconds,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,4,150,1,15.195832,301288,260749,66,34,40439,0.999668,0.998371,0.999160,0.998765,0.999287,0.000253,0.000840
1,6,150,1,15.387929,301288,260771,44,16,40457,0.999801,0.998914,0.999605,0.999259,0.999572,0.000169,0.000395
2,8,150,1,15.979540,301288,260777,38,9,40464,0.999844,0.999062,0.999778,0.999420,0.999665,0.000146,0.000222
3,6,250,1,24.339778,301288,260780,35,9,40464,0.999854,0.999136,0.999778,0.999457,0.999686,0.000134,0.000222
4,6,150,5,14.626370,301288,260770,45,22,40451,0.999778,0.998889,0.999456,0.999173,0.999522,0.000173,0.000544


## 15. Fit XGBoost cuối trên toàn Train P2

Chỉ chạy sau khi xem và duyệt cấu hình mục 14. Fit cấu hình đã chọn trên toàn Train P2, lưu model và training summary; không dùng Test để fit và không ghi đè artifact.

In [2]:
from pathlib import Path
import importlib
import sys

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_xgboost_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow XGBoost P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_xgboost_workflow
importlib.reload(p2_xgboost_workflow)

RUN_P2_XGBOOST_FINAL = True
if not RUN_P2_XGBOOST_FINAL:
    raise RuntimeError('Chưa duyệt để fit XGBoost trên toàn Train P2.')
p2_xgboost_validation_dir = p2_xgboost_workflow.latest_validation_run_dir(
    required_artifacts=('p2_xgboost_validation_selection.json',)
)
p2_xgboost_model_path = p2_xgboost_workflow.fit_final(p2_xgboost_validation_dir)
print(f'Model đã lưu: {p2_xgboost_model_path}')


RAM khả dụng: 7.22 GiB; đĩa trống: 66.21 GiB. Đây không phải giới hạn peak RAM thực tế.
RAM khả dụng: 7.20 GiB; đĩa trống: 65.77 GiB. Đây không phải giới hạn peak RAM thực tế.
Đã lưu XGBoost P2: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_xgboost_selected_model.joblib
Model đã lưu: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_xgboost_selected_model.joblib


## 16. Đánh giá XGBoost trên Test P2

Chỉ chạy sau khi model cuối mục 15 đã fit và được duyệt. Đánh giá Accuracy, Precision/Recall/F1 Attack, Macro-F1, FPR, FNR và confusion matrix. Test P2 đã được dùng lọc Train và đã xem trong thí nghiệm cũ; kết quả mô tả protocol, không dùng để chỉnh cấu hình.

In [3]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_xgboost_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow XGBoost P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_xgboost_workflow
importlib.reload(p2_xgboost_workflow)

RUN_P2_XGBOOST_TEST = True
if not RUN_P2_XGBOOST_TEST:
    raise RuntimeError('Chưa duyệt để đánh giá XGBoost trên Test P2.')
p2_xgboost_validation_dir = p2_xgboost_workflow.latest_validation_run_dir(
    required_artifacts=(
        'p2_xgboost_validation_selection.json',
        'p2_xgboost_selected_model.joblib',
        'p2_xgboost_selected_training.json',
    )
)
p2_xgboost_test_results = p2_xgboost_workflow.evaluate_test(p2_xgboost_validation_dir)
display(p2_xgboost_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,1037872,814865,172,141692,81143,0.863313,0.997885,0.364139,0.533572,0.726748,0.000211,0.635861


## 17. Chọn cấu hình MLP trên Validation P2

So sánh một hidden layer với `(32,)`, `(64,)`, `(128,)` neuron bằng `Pipeline(StandardScaler, MLPClassifier)`. Giữ ReLU/Adam, `alpha=0.0001`, `batch_size=1024`, `learning_rate_init=0.001`, `max_iter=200`, `early_stopping=False`, `random_state=42`. Chọn theo Macro-F1 trên Validation; nếu hòa chính xác, ưu tiên ít neuron hơn. Bảng và artifact ghi số vòng, loss và cảnh báo hội tụ; nếu có `ConvergenceWarning`, xem xét kết quả trước khi duyệt fit toàn Train.

Bật gate rồi chạy riêng cell này. Mỗi fit cần tối thiểu 4 GiB RAM khả dụng và 3 GiB đĩa trống; ước tính không bảo đảm peak RAM thực tế. Validation nằm trong Monday–Wednesday; không đọc các dòng Test để chọn cấu hình. Test đã được dùng lọc overlap khỏi Train và đã xem trong thí nghiệm cũ.

In [4]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_mlp_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow MLP P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_mlp_workflow
importlib.reload(p2_mlp_workflow)

RUN_P2_MLP_VALIDATION = True
if not RUN_P2_MLP_VALIDATION:
    raise RuntimeError('Chưa duyệt để chọn cấu hình MLP P2 trên Validation.')
p2_mlp_validation_dir = p2_mlp_workflow.latest_validation_run_dir()
p2_mlp_validation_results = p2_mlp_workflow.compare_validation(p2_mlp_validation_dir)
display(p2_mlp_validation_results)


Đã dùng cache đã xác minh: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_logistic_training_cache
Đang fit MLP P2 trên TrainInternal: hidden_layer_sizes=(32,)
Peak RAM ước tính: 1.53 GiB; RAM khả dụng: 7.21 GiB; đĩa trống: 66.21 GiB. Ước tính không bảo đảm peak RAM thực tế.
Đang fit MLP P2 trên TrainInternal: hidden_layer_sizes=(64,)
Peak RAM ước tính: 1.53 GiB; RAM khả dụng: 7.14 GiB; đĩa trống: 66.21 GiB. Ước tính không bảo đảm peak RAM thực tế.
Đang fit MLP P2 trên TrainInternal: hidden_layer_sizes=(128,)
Peak RAM ước tính: 1.53 GiB; RAM khả dụng: 7.12 GiB; đĩa trống: 66.21 GiB. Ước tính không bảo đảm peak RAM thực tế.
MLP P2 được chọn theo Macro-F1: {'activation': 'relu', 'alpha': 0.0001, 'batch_size': 1024, 'early_stopping': False, 'learning_rate_init': 0.001, 'max_iter': 200, 'random_state': 42, 'solver': 'adam', 'hidden_layer_sizes': [128]}


,hidden_units,fit_seconds,n_iter,final_loss,convergence_warnings,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,32,141.688784,83,0.005697,[],301288,260661,154,428,40045,0.998068,0.996169,0.989425,0.992786,0.995835,0.000590,0.010575
1,64,164.751990,73,0.005293,[],301288,260562,253,274,40199,0.998251,0.993746,0.993230,0.993488,0.996239,0.000970,0.006770
2,128,322.922811,98,0.005152,[],301288,260693,122,339,40134,0.998470,0.996969,0.991624,0.994290,0.996703,0.000468,0.008376


## 18. Fit MLP cuối trên toàn Train P2

Chỉ chạy sau khi xem và duyệt bảng Validation ở mục 17, đặc biệt là cảnh báo hội tụ. Fit pipeline `StandardScaler + MLPClassifier` với cấu hình được chọn trên toàn Train P2; scaler chỉ fit trên Train. Model và training summary được lưu riêng, không ghi đè artifact.

In [5]:
from pathlib import Path
import importlib
import sys

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_mlp_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow MLP P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_mlp_workflow
importlib.reload(p2_mlp_workflow)

RUN_P2_MLP_FINAL = True
if not RUN_P2_MLP_FINAL:
    raise RuntimeError('Chưa duyệt để fit MLP trên toàn Train P2.')
p2_mlp_validation_dir = p2_mlp_workflow.latest_validation_run_dir(
    required_artifacts=('p2_mlp_validation_selection.json',)
)
p2_mlp_model_path = p2_mlp_workflow.fit_final(p2_mlp_validation_dir)
print(f'Model đã lưu: {p2_mlp_model_path}')


Peak RAM ước tính: 1.88 GiB; RAM khả dụng: 5.84 GiB; đĩa trống: 65.77 GiB. Ước tính không bảo đảm peak RAM thực tế.
Đã lưu MLP pipeline P2: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_mlp_selected_pipeline.joblib
Model đã lưu: D:\HUST\GR2\NIDS-ML\Dataset\Processed\P2_Day_Disjoint\20261115_031802_3c13ca17\validation_runs\20261115_102529_6dbf209b\p2_mlp_selected_pipeline.joblib


## 19. Đánh giá MLP trên Test P2

Chỉ chạy sau khi model cuối mục 18 đã fit và training summary được xem, rồi được duyệt riêng. Báo cáo Accuracy, Precision/Recall/F1 Attack, Macro-F1, FPR, FNR và confusion matrix. Test P2 đã được dùng lọc Train và đã xem trong thí nghiệm cũ; kết quả chỉ mô tả protocol, không dùng để chỉnh cấu hình.

In [6]:
from pathlib import Path
import importlib
import sys
from IPython.display import display

project_root = next((root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'Notebooks' / 'p2_mlp_workflow.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy workflow MLP P2 trong project.')
helper_dir = project_root / 'Notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
import p2_logistic_workflow
importlib.reload(p2_logistic_workflow)
import p2_mlp_workflow
importlib.reload(p2_mlp_workflow)

RUN_P2_MLP_TEST = True
if not RUN_P2_MLP_TEST:
    raise RuntimeError('Chưa duyệt để đánh giá MLP trên Test P2.')
p2_mlp_validation_dir = p2_mlp_workflow.latest_validation_run_dir(
    required_artifacts=(
        'p2_mlp_validation_selection.json',
        'p2_mlp_selected_pipeline.joblib',
        'p2_mlp_selected_training.json',
    )
)
p2_mlp_test_results = p2_mlp_workflow.evaluate_test(p2_mlp_validation_dir)
display(p2_mlp_test_results)


,rows,TN,FP,FN,TP,accuracy,attack_precision,attack_recall,attack_f1,macro_f1,fpr,fnr
0,1037872,813533,1504,179198,43637,0.825892,0.966682,0.195827,0.325678,0.61286,0.001845,0.804173
